# Музыкальный датасет с семантическими признаками

Задача: собрать небольшой датасет треков (у меня 96 треков, 12 жанров), добавить к ним признаки (жанр, настроение, контекст, атмосфера, инструменты, темп и энергия, эпоха, похожие артисты) и разобраться, какие из них полезнее для рекомендательных систем.

Структура:
1. Сбор датасета (YouTube + MusicBrainz)
2. Набор признаков
3. Извлечение признаков (librosa + CLAP)
4. EDA
5. Эксперименты с похожестью треков
6. Выводы

Запуск: Colab с GPU, «Выполнить все». Данные пишутся на Google Drive в папку `music_dataset` (`audio/`, `data/`, `figures/`). Основные файлы на выходе: `data/tracks.csv`, `data/features.csv`, `data/clap_embeddings.npy`.

In [ ]:
%pip install -q "yt-dlp[default]" imageio-ffmpeg rapidfuzz librosa soundfile umap-learn transformers tqdm
%pip install -q deno

In [ ]:
import os, sys, json, time, re, subprocess, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)

IN_COLAB = 'google.colab' in sys.modules
USE_DRIVE = True

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/music_dataset')
else:
    BASE_DIR = Path('music_dataset').resolve()

AUDIO_DIR, DATA_DIR, FIG_DIR = BASE_DIR / 'audio', BASE_DIR / 'data', BASE_DIR / 'figures'
for d in (AUDIO_DIR, DATA_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

GENRE_ORDER = ['rock', 'pop', 'hip_hop', 'electronic', 'jazz', 'classical',
               'metal', 'rnb_soul', 'country', 'reggae', 'ambient', 'latin']
GENRE_RU = {'rock': 'рок', 'pop': 'поп', 'hip_hop': 'хип-хоп', 'electronic': 'электроника',
            'jazz': 'джаз', 'classical': 'классика', 'metal': 'метал', 'rnb_soul': 'R&B / соул',
            'country': 'кантри', 'reggae': 'регги', 'ambient': 'эмбиент', 'latin': 'латино'}

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

CAT = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6', '#e34948']
ACCENT = CAT[0]
sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 150, 'axes.titleweight': 'bold',
                     'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})

# жанров 12, а цветов 8, поэтому последние 4 жанра рисую треугольниками
def genre_style(g):
    i = GENRE_ORDER.index(g)
    return CAT[i % 8], ('o' if i < 8 else '^')

def savefig(name):
    plt.savefig(FIG_DIR / f'{name}.png', bbox_inches='tight')

def scatter_by_genre(ax, xy, genres, title):
    for g in GENRE_ORDER:
        m = np.asarray(genres) == g
        if m.any():
            c, mk = genre_style(g)
            ax.scatter(xy[m, 0], xy[m, 1], s=46, c=c, marker=mk, label=GENRE_RU[g],
                       edgecolors='white', linewidths=1)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)

## 1. Сбор датасета

**Источники.** Аудио и метаданные я беру с YouTube через `yt-dlp`: у почти всех известных треков там есть официальная загрузка (каналы `... - Topic`, VEVO или канал самого артиста). С YouTube же забираю длительность, количество просмотров и лайков, дату загрузки и теги. Год выпуска дополнительно сверяю с MusicBrainz, это открытая музыкальная база с API.

**Как отбирала треки.** Список составила вручную: 12 жанров, в каждом по 8 известных треков. Брала узнаваемые записи, чтобы на YouTube точно нашлась оригинальная версия, а не кавер. Внутри жанра старалась взять треки из разных десятилетий, иначе эпоха полностью совпадала бы с жанром.

Видео для каждого трека подбирается автоматически: ищу на YouTube и выбираю результат, в котором совпадают название и исполнитель, канал официальный и в заголовке нет слов cover, remix, live, karaoke и т. п. Потом спорные случаи проверяю глазами.

Из каждого трека вырезаю 30 секунд, начиная примерно с трети длины (там обычно уже куплет или припев, а не вступление). Сохраняю в WAV 48 кГц моно, потому что с такой частотой работает CLAP.

**Жанры:** рок, поп, хип-хоп, электроника, джаз, классика, метал, R&B/соул, кантри, регги, эмбиент, латино.

**Ограничения:**
- 8 треков на жанр, всего 96, чтобы классы были сбалансированы;
- один трек на исполнителя, иначе при поиске похожих треков модель находила бы просто того же артиста;
- у классики в качестве года стоит год написания произведения, а не год записи.

In [ ]:
import io
MAX_PER_GENRE, MAX_PER_ARTIST = 8, 1

SEED_CSV = """track_id,artist,title,genre,year,query
rock_01,Queen,Bohemian Rhapsody,rock,1975,
rock_02,Led Zeppelin,Stairway to Heaven,rock,1971,
rock_03,Nirvana,Smells Like Teen Spirit,rock,1991,
rock_04,The Rolling Stones,Paint It Black,rock,1966,
rock_05,Arctic Monkeys,Do I Wanna Know?,rock,2013,
rock_06,Radiohead,Creep,rock,1992,
rock_07,The Killers,Mr. Brightside,rock,2003,
rock_08,Foo Fighters,Everlong,rock,1997,
pop_01,Michael Jackson,Billie Jean,pop,1982,
pop_02,Madonna,Like a Prayer,pop,1989,
pop_03,Britney Spears,Toxic,pop,2003,
pop_04,Dua Lipa,Levitating,pop,2020,
pop_05,The Weeknd,Blinding Lights,pop,2019,
pop_06,Taylor Swift,Shake It Off,pop,2014,
pop_07,ABBA,Dancing Queen,pop,1976,
pop_08,Billie Eilish,bad guy,pop,2019,
hip_hop_01,Dr. Dre,Still D.R.E. (feat. Snoop Dogg),hip_hop,1999,
hip_hop_02,Eminem,Lose Yourself,hip_hop,2002,
hip_hop_03,Kendrick Lamar,HUMBLE.,hip_hop,2017,
hip_hop_04,The Notorious B.I.G.,Juicy,hip_hop,1994,
hip_hop_05,2Pac,California Love (feat. Dr. Dre),hip_hop,1995,
hip_hop_06,OutKast,Ms. Jackson,hip_hop,2000,
hip_hop_07,Run-DMC,It's Tricky,hip_hop,1986,
hip_hop_08,Travis Scott,SICKO MODE,hip_hop,2018,
electronic_01,Daft Punk,One More Time,electronic,2000,
electronic_02,Avicii,Levels,electronic,2011,
electronic_03,The Prodigy,Firestarter,electronic,1996,
electronic_04,deadmau5,Strobe,electronic,2009,
electronic_05,The Chemical Brothers,Hey Boy Hey Girl,electronic,1999,
electronic_06,Kraftwerk,The Model,electronic,1978,
electronic_07,Swedish House Mafia,Don't You Worry Child,electronic,2012,
electronic_08,Fred again..,Delilah (pull me out of this),electronic,2022,
jazz_01,Miles Davis,So What,jazz,1959,
jazz_02,The Dave Brubeck Quartet,Take Five,jazz,1959,
jazz_03,John Coltrane,Giant Steps,jazz,1960,
jazz_04,Louis Armstrong,What a Wonderful World,jazz,1967,
jazz_05,Ella Fitzgerald,Mack the Knife (Live in Berlin),jazz,1960,
jazz_06,Herbie Hancock,Cantaloupe Island,jazz,1964,
jazz_07,Chet Baker,My Funny Valentine,jazz,1954,
jazz_08,Kamasi Washington,Truth,jazz,2017,
classical_01,Ludwig van Beethoven,Symphony No. 5 in C minor: I. Allegro con brio,classical,1808,Beethoven Symphony No. 5 1st movement Allegro con brio
classical_02,Wolfgang Amadeus Mozart,Eine kleine Nachtmusik: I. Allegro,classical,1787,Mozart Eine kleine Nachtmusik I. Allegro
classical_03,Antonio Vivaldi,The Four Seasons - Spring: I. Allegro,classical,1725,Vivaldi Four Seasons Spring I. Allegro
classical_04,Johann Sebastian Bach,Cello Suite No. 1 in G major: Prelude,classical,1720,Bach Cello Suite No. 1 Prelude
classical_05,Claude Debussy,Clair de lune,classical,1905,Debussy Clair de lune piano
classical_06,Pyotr Ilyich Tchaikovsky,The Nutcracker: Dance of the Sugar Plum Fairy,classical,1892,Tchaikovsky Nutcracker Dance of the Sugar Plum Fairy
classical_07,Frédéric Chopin,"Nocturne in E-flat major, Op. 9 No. 2",classical,1832,Chopin Nocturne Op. 9 No. 2
classical_08,Erik Satie,Gymnopédie No. 1,classical,1888,Satie Gymnopedie No. 1
metal_01,Metallica,Master of Puppets,metal,1986,
metal_02,Black Sabbath,Paranoid,metal,1970,
metal_03,Iron Maiden,The Trooper,metal,1983,
metal_04,Slayer,Raining Blood,metal,1986,
metal_05,System of a Down,Chop Suey!,metal,2001,
metal_06,Rammstein,Du hast,metal,1997,
metal_07,Slipknot,Duality,metal,2004,
metal_08,Gojira,Stranded,metal,2017,
rnb_soul_01,Marvin Gaye,What's Going On,rnb_soul,1971,
rnb_soul_02,Aretha Franklin,Respect,rnb_soul,1967,
rnb_soul_03,Stevie Wonder,Superstition,rnb_soul,1972,
rnb_soul_04,Whitney Houston,I Wanna Dance with Somebody (Who Loves Me),rnb_soul,1987,
rnb_soul_05,Destiny's Child,Say My Name,rnb_soul,1999,
rnb_soul_06,Frank Ocean,Thinkin Bout You,rnb_soul,2012,
rnb_soul_07,SZA,Kill Bill,rnb_soul,2022,
rnb_soul_08,Alicia Keys,Fallin',rnb_soul,2001,
country_01,Johnny Cash,Ring of Fire,country,1963,
country_02,Dolly Parton,Jolene,country,1973,
country_03,Kenny Rogers,The Gambler,country,1978,
country_04,John Denver,"Take Me Home, Country Roads",country,1971,
country_05,Shania Twain,Man! I Feel Like a Woman!,country,1997,
country_06,Luke Combs,Fast Car,country,2023,
country_07,Chris Stapleton,Tennessee Whiskey,country,2015,
country_08,Willie Nelson,On the Road Again,country,1980,
reggae_01,Bob Marley & The Wailers,Three Little Birds,reggae,1977,
reggae_02,Peter Tosh,Legalize It,reggae,1976,
reggae_03,Jimmy Cliff,The Harder They Come,reggae,1972,
reggae_04,Toots & The Maytals,Pressure Drop,reggae,1969,
reggae_05,UB40,Red Red Wine,reggae,1983,
reggae_06,Damian Marley,Welcome to Jamrock,reggae,2005,
reggae_07,Burning Spear,Marcus Garvey,reggae,1975,
reggae_08,Chronixx,Here Comes Trouble,reggae,2014,
ambient_01,Brian Eno,An Ending (Ascent),ambient,1983,
ambient_02,Aphex Twin,Rhubarb,ambient,1994,Aphex Twin Rhubarb Selected Ambient Works Volume II
ambient_03,Boards of Canada,Roygbiv,ambient,1998,
ambient_04,Max Richter,On the Nature of Daylight,ambient,2004,
ambient_05,Marconi Union,Weightless,ambient,2011,
ambient_06,Hiroshi Yoshimura,Water Copy,ambient,1982,
ambient_07,Jon Hopkins,Abandon Window,ambient,2013,
ambient_08,Tycho,Awake,ambient,2014,
latin_01,Luis Fonsi,Despacito (feat. Daddy Yankee),latin,2017,
latin_02,Buena Vista Social Club,Chan Chan,latin,1997,
latin_03,Shakira,Hips Don't Lie (feat. Wyclef Jean),latin,2006,
latin_04,Santana,Oye Como Va,latin,1970,
latin_05,Bad Bunny,Tití Me Preguntó,latin,2022,
latin_06,Celia Cruz,La Vida Es Un Carnaval,latin,1998,
latin_07,Stan Getz & João Gilberto,The Girl from Ipanema,latin,1964,
latin_08,Gipsy Kings,Bamboléo,latin,1987,
"""
seed = pd.read_csv(io.StringIO(SEED_CSV), dtype={'query': str})
seed['query'] = seed['query'].fillna('')

assert seed.groupby('genre').size().max() <= MAX_PER_GENRE
assert seed.groupby('artist').size().max() <= MAX_PER_ARTIST
assert seed['track_id'].is_unique
print(len(seed), 'треков,', seed.genre.nunique(), 'жанров,', seed.artist.nunique(), 'исполнителей')
seed.groupby('genre').size().rename(index=GENRE_RU)

### Загрузка с YouTube
Если YouTube из Colab начнёт просить подтверждение («Sign in to confirm you're not a bot»), можно положить cookies из браузера в `cookies.txt` и указать путь в `COOKIES_FILE`. Другой вариант: запустить эту часть локально. Уже скачанные треки повторно не грузятся.

In [ ]:
import yt_dlp, imageio_ffmpeg
from rapidfuzz import fuzz
from tqdm.auto import tqdm

FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
COOKIES_FILE = None
CLIP_SEC = 30
CLIP_POS = 0.35
SLEEP_SEC = 1.5

BAD_WORDS = ['cover', 'karaoke', 'remix', 'live', 'reaction', 'lyrics', '8d', 'slowed', 'sped up',
             'nightcore', 'instrumental', 'tutorial', 'lesson', 'piano version', 'loop', '1 hour', '10 hours']

def ydl_opts(**extra):
    opts = dict(quiet=True, no_warnings=True, noplaylist=True, ffmpeg_location=FFMPEG)
    if COOKIES_FILE:
        opts['cookiefile'] = COOKIES_FILE
    opts.update(extra)
    return opts

def search_youtube(query, n=6):
    with yt_dlp.YoutubeDL(ydl_opts(extract_flat=True)) as ydl:
        res = ydl.extract_info(f'ytsearch{n}:{query}', download=False)
    return [e for e in (res.get('entries') or []) if e and e.get('id')]

def score_candidate(e, ref_text, artist, title):
    vt = (e.get('title') or '').lower()
    ch = (e.get('channel') or e.get('uploader') or '').lower()
    s = fuzz.token_set_ratio(ref_text.lower(), f'{ch} {vt}')
    if ch.endswith(' - topic'):
        s += 15
    if 'vevo' in ch or fuzz.partial_ratio(artist.lower(), ch) >= 85:
        s += 10
    if 'official' in vt:
        s += 5
    for w in BAD_WORDS:
        if w in vt and w not in title.lower():
            s -= 25
    d = e.get('duration') or 0
    if d and not (60 <= d <= 1200):
        s -= 40
    return s

def pick_video(row):
    if row.track_id in MANUAL_OVERRIDES:
        return {'id': MANUAL_OVERRIDES[row.track_id]}, 100.0
    query = row.query.strip() or f'{row.artist} - {row.title} official audio'
    ref = row.query.strip() or f'{row.artist} {row.title}'
    cands = search_youtube(query)
    if not cands:
        return None, None
    best = max(cands, key=lambda e: score_candidate(e, ref, row.artist, row.title))
    return best, float(score_candidate(best, ref, row.artist, row.title))

def download_clip(video_id, track_id):
    url = f'https://www.youtube.com/watch?v={video_id}'
    tmpl = str(AUDIO_DIR / f'_full_{track_id}.%(ext)s')
    with yt_dlp.YoutubeDL(ydl_opts(format='bestaudio[ext=m4a]/bestaudio/best', outtmpl=tmpl)) as ydl:
        info = ydl.extract_info(url, download=True)
    full = (info.get('requested_downloads') or [{}])[0].get('filepath')
    if not full or not Path(full).exists():
        full = str(next(AUDIO_DIR.glob(f'_full_{track_id}.*')))
    dur = float(info.get('duration') or 0)
    start = max(0.0, min(dur * CLIP_POS, dur - CLIP_SEC)) if dur > CLIP_SEC else 0.0
    out = AUDIO_DIR / f'{track_id}.wav'
    subprocess.run([FFMPEG, '-y', '-loglevel', 'error', '-ss', f'{start:.2f}', '-t', str(CLIP_SEC),
                    '-i', full, '-ac', '1', '-ar', '48000', str(out)], check=True)
    Path(full).unlink(missing_ok=True)
    return info, start

def youtube_record(row, info, start, score):
    vid = info['id']
    up = info.get('upload_date') or ''
    channel = info.get('channel') or info.get('uploader') or ''
    return {
        'track_id': row.track_id, 'artist': row.artist, 'title': row.title, 'genre': row.genre,
        'year': int(row.year), 'source': 'YouTube',
        'yt_video_id': vid, 'yt_url': f'https://www.youtube.com/watch?v={vid}',
        'clip_url': f'https://www.youtube.com/watch?v={vid}&t={int(start)}s',
        'yt_title': info.get('title'), 'yt_channel': channel,
        'is_topic_channel': channel.lower().endswith(' - topic'),
        'duration_sec': info.get('duration'), 'view_count': info.get('view_count'),
        'like_count': info.get('like_count'),
        'upload_date': f'{up[:4]}-{up[4:6]}-{up[6:]}' if len(up) == 8 else None,
        'yt_categories': '|'.join(info.get('categories') or []),
        'yt_tags': '|'.join((info.get('tags') or [])[:10]),
        'match_score': score, 'clip_start_sec': round(start, 2), 'clip_sec': CLIP_SEC,
        'audio_file': f'audio/{row.track_id}.wav',
    }

In [ ]:
# сюда вписываю id правильного видео, если поиск ошибся: {'rock_01': '...'}
MANUAL_OVERRIDES = {}

CACHE_PATH = DATA_DIR / 'youtube_meta_cache.json'
cache = json.loads(CACHE_PATH.read_text(encoding='utf-8')) if CACHE_PATH.exists() else {}
failed = []

for row in tqdm(list(seed.itertuples(index=False))):
    wav = AUDIO_DIR / f'{row.track_id}.wav'
    cached = cache.get(row.track_id)
    if cached and wav.exists() and (row.track_id not in MANUAL_OVERRIDES
                                    or cached['yt_video_id'] == MANUAL_OVERRIDES[row.track_id]):
        continue
    try:
        cand, score = pick_video(row)
        if cand is None:
            raise RuntimeError('ничего не найдено')
        info, start = download_clip(cand['id'], row.track_id)
        cache[row.track_id] = youtube_record(row, info, start, score)
        CACHE_PATH.write_text(json.dumps(cache, ensure_ascii=False, indent=1), encoding='utf-8')
    except Exception as e:
        failed.append((row.track_id, f'{type(e).__name__}: {str(e)[:200]}'))
    time.sleep(SLEEP_SEC)

tracks = pd.DataFrame([cache[t] for t in seed.track_id if t in cache and (AUDIO_DIR / f'{t}.wav').exists()])
print(f'Загружено {len(tracks)} из {len(seed)}')
for t, err in failed:
    print(t, err)

Видео с самой низкой оценкой совпадения, их проверяю вручную.

In [ ]:
tracks['needs_review'] = tracks['match_score'] < 60
tracks.sort_values('match_score')[['track_id', 'artist', 'title', 'yt_title', 'yt_channel', 'match_score', 'yt_url']].head(15)

### Сверка года с MusicBrainz
Для каждого трека (кроме классики) беру самую раннюю дату релиза. Основным оставляю год из своего списка, а треки с расхождением больше 2 лет смотрю отдельно. У MusicBrainz ограничение в 1 запрос в секунду, поэтому ставлю паузу.

In [ ]:
import requests
ENABLE_MUSICBRAINZ = True
MB_HEADERS = {'User-Agent': 'MusicFeaturesStudy/1.0 (student project)'}

def mb_first_year(artist, title):
    title = re.sub(r'\s*\((feat|ft)\..*?\)', '', title, flags=re.I).replace('"', '')
    artist = artist.replace('"', '')
    q = f'recording:"{title}" AND artist:"{artist}"'
    r = requests.get('https://musicbrainz.org/ws/2/recording', headers=MB_HEADERS, timeout=20,
                     params={'query': q, 'fmt': 'json', 'limit': 25})
    r.raise_for_status()
    years = [int(rec['first-release-date'][:4]) for rec in r.json().get('recordings', [])
             if rec.get('score', 0) >= 90 and str(rec.get('first-release-date', ''))[:4].isdigit()]
    return min(years) if years else None

mb_cache_path = DATA_DIR / 'musicbrainz_cache.json'
mb_cache = json.loads(mb_cache_path.read_text(encoding='utf-8')) if mb_cache_path.exists() else {}
if ENABLE_MUSICBRAINZ:
    for row in tqdm(list(tracks.itertuples(index=False))):
        if row.genre == 'classical' or row.track_id in mb_cache:
            continue
        try:
            mb_cache[row.track_id] = mb_first_year(row.artist, row.title)
        except Exception:
            mb_cache[row.track_id] = None
        time.sleep(1.1)
    mb_cache_path.write_text(json.dumps(mb_cache), encoding='utf-8')

tracks['year_musicbrainz'] = tracks['track_id'].map(mb_cache)
diff = (tracks['year'] - tracks['year_musicbrainz']).abs()
print('Найдено в MusicBrainz:', int(tracks.year_musicbrainz.notna().sum()))
tracks.loc[diff > 2, ['artist', 'title', 'year', 'year_musicbrainz']]

In [ ]:
def era_of(y):
    if y < 1900: return 'до 1900 (классика)'
    if y < 1960: return '1900–1959'
    if y < 1980: return '1960–1979'
    if y < 2000: return '1980–1999'
    if y < 2010: return '2000–2009'
    return '2010+'

tracks['decade'] = (tracks['year'] // 10 * 10).astype(int)
tracks['era'] = tracks['year'].apply(era_of)
tracks['log_views'] = np.log10(tracks['view_count'].fillna(0).astype(float) + 1)

tracks.to_csv(DATA_DIR / 'tracks.csv', index=False)
print(tracks.shape)
tracks.groupby('genre').agg(n=('track_id', 'size'), year_min=('year', 'min'), year_max=('year', 'max'),
                            views_median=('view_count', 'median'), topic_share=('is_topic_channel', 'mean')).rename(index=GENRE_RU)

In [ ]:
tracks[['track_id', 'artist', 'title', 'genre', 'year', 'yt_channel', 'duration_sec', 'view_count', 'clip_url']].head(10)

## 2. Какие признаки извлекаю

Признаки я разделила на несколько групп по тому, откуда они берутся.

**1. Аудио-эмбеддинг CLAP (512 чисел).** Плотное описание того, как звучит трек: тембр, аранжировка, продакшн. Его удобно использовать для поиска похожих треков. Главный плюс: эмбеддинг есть у любого трека, даже у нового, по которому ещё нет прослушиваний (проблема холодного старта).

**2. Семантические теги** (получаю zero-shot через CLAP):
- настроение: energetic, calm, happy, sad, aggressive, romantic, dreamy, dark;
- контекст: тренировка, вечеринка, работа/учёба, отдых, сон, в дорогу, романтический вечер;
- атмосфера: эпичная, камерная, грувовая, загадочная, ностальгическая, праздничная;
- инструменты (есть/нет): электрогитара, акустическая гитара, фортепиано, синтезатор, струнные, духовые/саксофон, ударные;
- есть ли вокал.

Эти признаки понятны человеку. По ним можно объяснить рекомендацию («похоже по настроению») и собрать плейлист под ситуацию. Пользователи часто ищут музыку именно так: «что-нибудь бодрое для пробежки», а не по жанру. Метал и электроника могут одинаково подходить для тренировки.

**3. Сигнальные признаки** (librosa): темп, громкость (RMS), динамический диапазон, яркость (спектральный центроид), шумность, плотность атак, выраженность ритма, доля ударных, лад (мажор/минор). Они считаются формулами, без нейросетей, поэтому им можно доверять. Темп и энергия напрямую связаны с контекстами вроде тренировки или сна. Ещё по ним можно проверить, насколько адекватны теги CLAP.

**4. Метаданные:** жанр, год, десятилетие, эпоха, длительность, популярность (логарифм просмотров). Эпоха нужна для тематических подборок вроде «хиты 80-х», популярность — чтобы балансировать выдачу между хитами и менее известными треками. Жанр я задала сама при отборе, поэтому использую его как эталон для проверки, а не как признак.

**5. Похожие артисты.** Считаю по ближайшим трекам в пространстве эмбеддингов (этап 5). Можно дополнительно спросить LLM, но это опционально.

CLAP тоже умеет предсказывать жанр. Я его считаю только чтобы проверить, насколько модель вообще справляется, и в признаки для похожести не включаю: иначе оценка по жанру была бы нечестной.

Для zero-shot у каждого класса по 2–3 текстовых шаблона, их эмбеддинги усредняются. Так результат меньше зависит от конкретной формулировки.

In [ ]:
FEATURE_SCHEMA = {
    'genre_clap': {
        'type': 'multiclass', 'role': 'validation',
        'templates': ['{} music', 'this is a {} song', 'a {} track'],
        'labels': {'rock': 'rock', 'pop': 'pop', 'hip_hop': 'hip hop rap', 'electronic': 'electronic dance',
                   'jazz': 'jazz', 'classical': 'classical orchestral', 'metal': 'heavy metal',
                   'rnb_soul': 'r&b soul', 'country': 'country', 'reggae': 'reggae',
                   'ambient': 'ambient', 'latin': 'latin'},
    },
    'mood': {
        'type': 'multiclass', 'role': 'feature',
        'templates': ['{} music', 'a {} song', 'this song sounds {}'],
        'labels': {'energetic': 'energetic and powerful', 'calm': 'calm and peaceful',
                   'happy': 'happy and cheerful', 'sad': 'sad and melancholic',
                   'aggressive': 'aggressive and angry', 'romantic': 'romantic and tender',
                   'dreamy': 'dreamy and ethereal', 'dark': 'dark and ominous'},
    },
    'context': {
        'type': 'multiclass', 'role': 'feature',
        'templates': ['music for {}', 'a song perfect for {}', 'a playlist for {}'],
        'labels': {'workout': 'a workout at the gym', 'party': 'a dance party',
                   'focus_work': 'focusing on work or studying', 'relaxation': 'relaxing after a long day',
                   'sleep': 'falling asleep', 'road_trip': 'a road trip in the car',
                   'romantic_evening': 'a romantic dinner'},
    },
    'atmosphere': {
        'type': 'multiclass', 'role': 'feature',
        'templates': ['music with an {} atmosphere', 'a song that feels {}'],
        'labels': {'epic': 'epic and cinematic', 'intimate': 'intimate and personal',
                   'groovy': 'groovy and funky', 'mysterious': 'mysterious and hypnotic',
                   'nostalgic': 'nostalgic and retro', 'festive': 'festive and celebratory'},
    },
    'instrument': {
        'type': 'binary', 'role': 'feature',
        'labels': {
            'electric_guitar': ['music with loud electric guitar', 'music without electric guitar'],
            'acoustic_guitar': ['music with acoustic guitar', 'music without acoustic guitar'],
            'piano': ['music with piano', 'music without piano'],
            'synthesizer': ['music with synthesizers', 'music without synthesizers'],
            'strings': ['music with orchestral strings, violins and cellos', 'music without string instruments'],
            'brass_sax': ['music with brass instruments or saxophone', 'music without brass or saxophone'],
            'drums': ['music with a prominent drum beat', 'music without drums'],
        },
    },
    'vocals': {
        'type': 'binary', 'role': 'feature',
        'labels': {'vocals': ['a song with a singer singing vocals', 'instrumental music without vocals']},
    },
}

SIGNAL_FEATURES = {
    'tempo_bpm': 'темп, BPM',
    'rms_mean': 'средняя громкость',
    'rms_dynamic_range_db': 'динамический диапазон, дБ',
    'spectral_centroid_hz': 'яркость (спектральный центроид), Гц',
    'spectral_flatness': 'шумность спектра',
    'onset_rate': 'атак в секунду',
    'pulse_clarity': 'выраженность ритма (0..1)',
    'percussive_ratio': 'доля энергии ударных',
    'zcr_mean': 'zero crossing rate',
    'mode_major': 'мажор (1) / минор (0)',
    'key_strength': 'уверенность в тональности',
}

META_FEATURES = {
    'genre': 'жанр (эталон)', 'year': 'год', 'decade': 'десятилетие', 'era': 'эпоха',
    'duration_sec': 'длительность', 'log_views': 'log10 просмотров', 'is_topic_channel': 'официальный канал лейбла',
}

schema = {'semantic': FEATURE_SCHEMA, 'signal': SIGNAL_FEATURES, 'meta': META_FEATURES,
          'embedding': {'clap_audio': 'laion/larger_clap_music, 512'}}
(DATA_DIR / 'feature_schema.json').write_text(json.dumps(schema, ensure_ascii=False, indent=2), encoding='utf-8')

rows = []
for cat, spec in FEATURE_SCHEMA.items():
    for lab, txt in spec['labels'].items():
        rows.append({'группа': 'семантика (CLAP)', 'признак': cat, 'класс': lab, 'роль': spec['role'],
                     'описание': txt if isinstance(txt, str) else ' / '.join(txt)})
rows += [{'группа': 'сигнал (librosa)', 'признак': k, 'класс': '', 'роль': 'feature', 'описание': v} for k, v in SIGNAL_FEATURES.items()]
rows += [{'группа': 'метаданные', 'признак': k, 'класс': '', 'роль': 'target' if k == 'genre' else 'feature', 'описание': v} for k, v in META_FEATURES.items()]
feature_table = pd.DataFrame(rows)
print('Признаков:', len(feature_table), '+ эмбеддинг')
feature_table

## 3. Пайплайн извлечения признаков

Схема такая: из каждого 30-секундного фрагмента считаю сигнальные признаки через librosa. Параллельно режу фрагмент на 3 окна по 10 секунд, прогоняю их через аудио-энкодер CLAP и усредняю, это и есть эмбеддинг трека. Потом сравниваю эмбеддинг с текстовыми эмбеддингами классов («energetic music», «music for a workout» и т. д.) и через softmax получаю вероятности настроения, контекста и атмосферы. Для инструментов и вокала сравниваю две фразы, «с X» и «без X».

**Что использую:**
- **CLAP** (`laion/larger_clap_music` с HuggingFace). Модель от LAION, обучена сопоставлять аудио и текст, эта версия дообучена на музыке. Из неё беру и эмбеддинги, и теги.
- **librosa** для темпа, громкости, спектральных признаков, разделения на гармоническую и ударную часть и хромаграммы. Тональность определяю по профилям Крумхансла.
- По желанию **LLM** (Anthropic API) для списка похожих артистов. По умолчанию выключено, нужен ключ.

**Почему CLAP.** Размеченных данных нет, а обучать классификатор настроения на 96 треках бессмысленно. CLAP позволяет задать классы просто текстом. Кроме того, теги считаются по самому звуку, а не по названию, поэтому подход сработает и для неизвестного трека. Окна по 10 секунд нужны, потому что модель обучалась на таких фрагментах.

Ещё одна деталь: CLAP склонен отдавать предпочтение некоторым классам почти для всех треков. Поэтому перед softmax я вычитаю из сходств среднее по датасету для каждого класса. Метки без этой поправки тоже сохраняю (`*_top_raw`), чтобы сравнить.

**Какие альтернативы рассматривала:**
- Essentia с моделями MTG (жанр, настроение, danceability). Классификаторы хорошие, но таксономии фиксированные, контекстов нет, и установка сложнее.
- MERT. Сильные эмбеддинги, но без текстовой части, теги пришлось бы учить на разметке.
- PANNs / AST (AudioSet). Хорошо находят инструменты, но с настроением справляются слабо.
- Только LLM по названию трека. Просто, но модель не слушает музыку, может выдумывать и ничего не знает о новых треках.
- Только librosa. Прозрачно, но семантику не ловит. Этот вариант я всё равно сравниваю на этапе 5.

In [ ]:
import librosa
from tqdm.auto import tqdm

tracks = pd.read_csv(DATA_DIR / 'tracks.csv')
schema = json.loads((DATA_DIR / 'feature_schema.json').read_text(encoding='utf-8'))
FEATURE_SCHEMA = schema['semantic']
tracks = tracks[tracks['track_id'].apply(lambda t: (AUDIO_DIR / f'{t}.wav').exists())].reset_index(drop=True)
print('Треков с аудио:', len(tracks))

SR_CLAP, SR_SIG = 48000, 22050
WIN_SEC, N_WIN = 10, 3

def load_audio(track_id):
    y, _ = librosa.load(str(AUDIO_DIR / f'{track_id}.wav'), sr=SR_CLAP, mono=True)
    return y.astype(np.float32)

def make_windows(y, sr=SR_CLAP, win_sec=WIN_SEC, n=N_WIN):
    L = int(win_sec * sr)
    if len(y) < L * n:
        y = np.pad(y, (0, L * n - len(y)))
    starts = np.linspace(0, len(y) - L, n).astype(int)
    return [y[s:s + L] for s in starts]

### 3.1 Признаки librosa

In [ ]:
MAJOR_PROFILE = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
MINOR_PROFILE = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17])
PITCHES = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

def estimate_key(chroma_mean):
    best = (-2, None, None)
    for i in range(12):
        for prof, mode in ((MAJOR_PROFILE, 1), (MINOR_PROFILE, 0)):
            r = np.corrcoef(chroma_mean, np.roll(prof, i))[0, 1]
            if r > best[0]:
                best = (r, i, mode)
    r, i, mode = best
    return f"{PITCHES[i]} {'major' if mode else 'minor'}", mode, float(r)

def signal_features(y48):
    y = librosa.resample(y48, orig_sr=SR_CLAP, target_sr=SR_SIG)
    sr, hop = SR_SIG, 512
    dur = len(y) / sr
    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop)
    tempo, _ = librosa.beat.beat_track(onset_envelope=onset_env, sr=sr, hop_length=hop)
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    onsets = librosa.onset.onset_detect(onset_envelope=onset_env, sr=sr, hop_length=hop)
    # ритмичность: пик автокорреляции огибающей атак в диапазоне 40-200 BPM
    oe = onset_env - onset_env.mean()
    ac = librosa.autocorrelate(oe)
    ac = ac / (ac[0] + 1e-9)
    lag_min, lag_max = int(60 / 200 * sr / hop), int(60 / 40 * sr / hop)
    pulse = float(np.max(ac[lag_min:lag_max])) if len(ac) > lag_max else np.nan
    y_harm, y_perc = librosa.effects.hpss(y)
    chroma = librosa.feature.chroma_cqt(y=y_harm, sr=sr, hop_length=hop).mean(axis=1)
    key, mode, key_strength = estimate_key(chroma)
    p95, p5 = np.percentile(rms, 95), np.percentile(rms, 5)
    return {
        'tempo_bpm': float(np.atleast_1d(tempo)[0]),
        'rms_mean': float(rms.mean()),
        'rms_dynamic_range_db': float(20 * np.log10((p95 + 1e-6) / (p5 + 1e-6))),
        'spectral_centroid_hz': float(librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0].mean()),
        'spectral_flatness': float(librosa.feature.spectral_flatness(y=y, hop_length=hop)[0].mean()),
        'onset_rate': len(onsets) / dur,
        'pulse_clarity': pulse,
        'percussive_ratio': float(np.sum(y_perc ** 2) / (np.sum(y_harm ** 2) + np.sum(y_perc ** 2) + 1e-9)),
        'zcr_mean': float(librosa.feature.zero_crossing_rate(y, hop_length=hop)[0].mean()),
        'key': key, 'mode_major': mode, 'key_strength': key_strength,
    }

audio_cache, sig_rows = {}, []
for t in tqdm(tracks['track_id']):
    y = load_audio(t)
    audio_cache[t] = y
    sig_rows.append({'track_id': t, **signal_features(y)})
signal_df = pd.DataFrame(sig_rows)
signal_df.describe().T.round(3)

### 3.2 CLAP
Лучше запускать с GPU (Среда выполнения → Сменить среду → T4 GPU), на CPU тоже работает, но медленнее.

In [ ]:
import torch
from transformers import ClapModel, ClapProcessor

CLAP_ID = 'laion/larger_clap_music'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
clap_model = ClapModel.from_pretrained(CLAP_ID).to(device).eval()
clap_proc = ClapProcessor.from_pretrained(CLAP_ID)

# в новых версиях transformers get_*_features возвращает не тензор, а объект
def _embeds(out, kind):
    if torch.is_tensor(out):
        return out
    for attr in (f'{kind}_embeds', 'pooler_output'):
        v = getattr(out, attr, None)
        if v is not None:
            return v
    raise TypeError(type(out))

@torch.no_grad()
def clap_embed_audio(waves, batch_size=8):
    out = []
    for i in range(0, len(waves), batch_size):
        inp = clap_proc.feature_extractor(waves[i:i + batch_size], sampling_rate=SR_CLAP, return_tensors='pt')
        inp = {k: v.to(device) for k, v in inp.items()}
        e = _embeds(clap_model.get_audio_features(**inp), 'audio')
        out.append(torch.nn.functional.normalize(e, dim=-1).cpu().numpy())
    return np.concatenate(out)

@torch.no_grad()
def clap_embed_text(texts):
    inp = clap_proc.tokenizer(texts, padding=True, return_tensors='pt')
    inp = {k: v.to(device) for k, v in inp.items() if k in ('input_ids', 'attention_mask')}
    e = _embeds(clap_model.get_text_features(**inp), 'text')
    return torch.nn.functional.normalize(e, dim=-1).cpu().numpy()

try:
    LOGIT_SCALE = float(clap_model.logit_scale_a.exp().item())
except Exception:
    LOGIT_SCALE = 30.0
print(device, LOGIT_SCALE)

In [ ]:
win_list, owner = [], []
for i, t in enumerate(tracks['track_id']):
    for w in make_windows(audio_cache[t]):
        win_list.append(w); owner.append(i)
win_emb = clap_embed_audio(win_list)
owner = np.array(owner)
E = np.stack([win_emb[owner == i].mean(0) for i in range(len(tracks))])
E = E / np.linalg.norm(E, axis=1, keepdims=True)
win_consistency = np.array([(win_emb[owner == i] @ E[i]).mean() for i in range(len(tracks))])
print(E.shape, 'средняя близость окон к треку:', win_consistency.mean().round(3))

### 3.3 Zero-shot теги

In [ ]:
def softmax(z, axis=1):
    z = z - z.max(axis=axis, keepdims=True)
    ez = np.exp(z)
    return ez / ez.sum(axis=axis, keepdims=True)

def text_prototypes(spec):
    protos = []
    for lab, txt in spec['labels'].items():
        T = clap_embed_text([tpl.format(txt) for tpl in spec['templates']])
        p = T.mean(0)
        protos.append(p / np.linalg.norm(p))
    return np.stack(protos)

def zero_shot_multiclass(E, spec, debias=True):
    sims = E @ text_prototypes(spec).T
    raw = softmax(LOGIT_SCALE * sims)
    if debias:
        sims = sims - sims.mean(0, keepdims=True)
    return softmax(LOGIT_SCALE * sims), raw

def zero_shot_binary(E, spec):
    out = {}
    for lab, (pos, neg) in spec['labels'].items():
        T = clap_embed_text([pos, neg])
        out[lab] = softmax(LOGIT_SCALE * (E @ T.T))[:, 0]
    return out

sem = pd.DataFrame({'track_id': tracks['track_id']})
prefix = {'genre_clap': 'clapgenre', 'mood': 'mood', 'context': 'ctx', 'atmosphere': 'atm'}
for cat, pfx in prefix.items():
    spec = FEATURE_SCHEMA[cat]
    labels = list(spec['labels'])
    P, P_raw = zero_shot_multiclass(E, spec)
    for j, lab in enumerate(labels):
        sem[f'{pfx}_{lab}'] = P[:, j]
    order = np.argsort(-P, axis=1)
    sem[f'{pfx}_top'] = [labels[k] for k in order[:, 0]]
    sem[f'{pfx}_top2'] = [labels[k] for k in order[:, 1]]
    sem[f'{pfx}_top_conf'] = P.max(1)
    sem[f'{pfx}_top_raw'] = [labels[k] for k in P_raw.argmax(1)]

for lab, p in zero_shot_binary(E, FEATURE_SCHEMA['instrument']).items():
    sem[f'inst_{lab}'] = p
sem['vocals_prob'] = zero_shot_binary(E, FEATURE_SCHEMA['vocals'])['vocals']
sem['has_vocals'] = sem['vocals_prob'] > 0.5
sem[['track_id', 'mood_top', 'ctx_top', 'ctx_top2', 'atm_top', 'clapgenre_top', 'vocals_prob']].head(12)

### 3.4 Похожие артисты через LLM (необязательно)
Здесь LLM отвечает по своим знаниям о треке, а не по звуку. Включается флагом `USE_LLM`, ключ в Colab лежит в Secrets под именем `ANTHROPIC_API_KEY`.

In [ ]:
USE_LLM = False
LLM_MODEL = 'claude-haiku-5-5'
llm_path = DATA_DIR / 'llm_tags.json'

if USE_LLM:
    %pip install -q anthropic
    import anthropic
    if IN_COLAB:
        from google.colab import userdata
        os.environ.setdefault('ANTHROPIC_API_KEY', userdata.get('ANTHROPIC_API_KEY'))
    client = anthropic.Anthropic()
    llm = json.loads(llm_path.read_text(encoding='utf-8')) if llm_path.exists() else {}
    for row in tqdm(list(tracks.itertuples(index=False))):
        if row.track_id in llm:
            continue
        prompt = (f'Track: "{row.title}" by {row.artist} ({row.year}). Reply with JSON only: '
                  '{"similar_artists": [3 artists], "subgenre": "...", "mood_tags": [3 words]}')
        msg = client.messages.create(model=LLM_MODEL, max_tokens=200, messages=[{'role': 'user', 'content': prompt}])
        txt = msg.content[0].text
        try:
            llm[row.track_id] = json.loads(txt[txt.find('{'): txt.rfind('}') + 1])
        except Exception:
            llm[row.track_id] = {}
    llm_path.write_text(json.dumps(llm, ensure_ascii=False, indent=1), encoding='utf-8')

if llm_path.exists():
    llm = json.loads(llm_path.read_text(encoding='utf-8'))
    sem['llm_similar_artists'] = sem['track_id'].map(lambda t: '|'.join(llm.get(t, {}).get('similar_artists', [])))
    sem['llm_subgenre'] = sem['track_id'].map(lambda t: llm.get(t, {}).get('subgenre'))

### 3.5 Сохранение и быстрая проверка
Проверяю две вещи: насколько хорошо CLAP угадывает жанр (случайное угадывание даёт 1/12) и согласуются ли теги с сигнальными признаками. Например, у «энергичных» треков должна быть выше громкость.

In [ ]:
features = tracks.merge(signal_df, on='track_id').merge(sem, on='track_id')
features['clap_window_consistency'] = win_consistency
features.to_csv(DATA_DIR / 'features.csv', index=False)
np.save(DATA_DIR / 'clap_embeddings.npy', E)
pd.Series(tracks['track_id']).to_csv(DATA_DIR / 'clap_embeddings_ids.csv', index=False)
print(features.shape, E.shape)

acc = (features['clapgenre_top'] == features['genre']).mean()
acc_raw = (features['clapgenre_top_raw'] == features['genre']).mean()
top2 = ((features['clapgenre_top'] == features['genre']) | (features['clapgenre_top2'] == features['genre'])).mean()
print(f'жанр zero-shot: acc = {acc:.2f}, без поправки {acc_raw:.2f}, top-2 = {top2:.2f}')

from scipy.stats import spearmanr
checks = [('mood_energetic', 'rms_mean'), ('mood_calm', 'rms_mean'), ('ctx_workout', 'tempo_bpm'),
          ('ctx_sleep', 'onset_rate'), ('inst_drums', 'percussive_ratio'), ('mood_happy', 'mode_major')]
pd.DataFrame([{'тег': a, 'сигнал': b, 'spearman': round(spearmanr(features[a], features[b], nan_policy='omit')[0], 3)}
              for a, b in checks])

## 4. EDA
Смотрю на состав датасета, распределения тегов, связь тегов с жанрами, сигнальные признаки по жанрам и корреляции. Отдельно проверяю, как CLAP угадывает жанр. Графики сохраняются в `figures/`.

In [ ]:
features = pd.read_csv(DATA_DIR / 'features.csv')
MOODS = [c[5:] for c in features.columns if c.startswith('mood_') and not c.startswith(('mood_top'))]
CTXS = [c[4:] for c in features.columns if c.startswith('ctx_') and not c.startswith('ctx_top')]
ATMS = [c[4:] for c in features.columns if c.startswith('atm_') and not c.startswith('atm_top')]
INSTS = [c[5:] for c in features.columns if c.startswith('inst_')]
SIGNAL = ['tempo_bpm', 'rms_mean', 'rms_dynamic_range_db', 'spectral_centroid_hz', 'spectral_flatness',
          'onset_rate', 'pulse_clarity', 'percussive_ratio', 'zcr_mean', 'mode_major', 'key_strength']
MOOD_RU = {'energetic': 'энергичное', 'calm': 'спокойное', 'happy': 'радостное', 'sad': 'грустное',
           'aggressive': 'агрессивное', 'romantic': 'романтичное', 'dreamy': 'мечтательное', 'dark': 'мрачное'}
CTX_RU = {'workout': 'тренировка', 'party': 'вечеринка', 'focus_work': 'работа / учёба', 'relaxation': 'отдых',
          'sleep': 'сон', 'road_trip': 'в дорогу', 'romantic_evening': 'романтический вечер'}
ATM_RU = {'epic': 'эпичная', 'intimate': 'камерная', 'groovy': 'грувовая', 'mysterious': 'загадочная',
          'nostalgic': 'ностальгическая', 'festive': 'праздничная'}
INST_RU = {'electric_guitar': 'электрогитара', 'acoustic_guitar': 'акуст. гитара', 'piano': 'фортепиано',
           'synthesizer': 'синтезатор', 'strings': 'струнные', 'brass_sax': 'духовые / сакс', 'drums': 'ударные'}
genre_order = [g for g in GENRE_ORDER if g in set(features.genre)]
features.shape

### 4.1 Состав

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
cnt = features['genre'].value_counts().reindex(genre_order)
axes[0].barh([GENRE_RU[g] for g in genre_order][::-1], cnt.values[::-1], color=ACCENT, height=0.6)
axes[0].set_title('Треков по жанрам'); axes[0].set_xlabel('треков')

dec = features['decade'].value_counts().sort_index()
axes[1].bar(dec.index.astype(str), dec.values, color=ACCENT, width=0.7)
axes[1].set_title('Треков по десятилетиям'); axes[1].tick_params(axis='x', rotation=60)

d = features[features.genre != 'classical']
for g in genre_order:
    if g == 'classical':
        continue
    c, mk = genre_style(g)
    m = d.genre == g
    axes[2].scatter(d.loc[m, 'year'], d.loc[m, 'log_views'], c=c, marker=mk, s=40, label=GENRE_RU[g],
                    edgecolors='#fcfcfb', linewidths=1)
axes[2].set_title('Популярность и год (без классики)'); axes[2].set_xlabel('год'); axes[2].set_ylabel('log10(просмотры)')
axes[2].legend(fontsize=7, ncol=2, loc='lower right')
plt.tight_layout(); savefig('01_dataset_composition'); plt.show()

### 4.2 Настроения, контексты, атмосфера
Слева количество треков, у которых класс получился первым, справа средняя вероятность класса. Второй график честнее, потому что трек может быть одновременно и энергичным, и радостным.

In [ ]:
def dist_plot(prefix, labels, ru, title, name):
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
    top = features[f'{prefix}_top'].value_counts().reindex(labels, fill_value=0).sort_values()
    axes[0].barh([ru[l] for l in top.index], top.values, color=ACCENT, height=0.6)
    axes[0].set_title(f'{title}: top-1 класс'); axes[0].set_xlabel('треков')
    for y, v in enumerate(top.values):
        axes[0].text(v + 0.3, y, str(v), va='center', fontsize=8, color='#52514e')
    soft = features[[f'{prefix}_{l}' for l in labels]].mean().sort_values()
    axes[1].barh([ru[c.split('_', 1)[1]] for c in soft.index], soft.values, color=ACCENT, height=0.6)
    axes[1].set_title(f'{title}: средняя вероятность'); axes[1].set_xlabel('P')
    plt.tight_layout(); savefig(name); plt.show()

dist_plot('mood', MOODS, MOOD_RU, 'Настроение', '02_mood_distribution')
dist_plot('ctx', CTXS, CTX_RU, 'Контекст', '03_context_distribution')
dist_plot('atm', ATMS, ATM_RU, 'Атмосфера', '04_atmosphere_distribution')

# сравнение с поправкой и без
cmp = pd.DataFrame({'с дебиасингом': features['mood_top'].value_counts(),
                    'без дебиасинга': features['mood_top_raw'].value_counts()}).reindex(MOODS).fillna(0).astype(int)
cmp.index = [MOOD_RU[m] for m in cmp.index]
cmp.T

### 4.3 Теги по жанрам
Средняя вероятность класса внутри жанра. Если теги работают, должно быть видно очевидное: метал агрессивный, эмбиент для сна и т. д.

In [ ]:
def genre_heatmap(prefix, labels, ru, title, name, ax):
    M = features.groupby('genre')[[f'{prefix}_{l}' for l in labels]].mean().reindex(genre_order)
    M.index = [GENRE_RU[g] for g in M.index]; M.columns = [ru[l] for l in labels]
    sns.heatmap(M, cmap='Blues', ax=ax, annot=True, fmt='.2f', annot_kws={'size': 7},
                linewidths=1, linecolor='#fcfcfb', cbar_kws={'shrink': 0.7})
    ax.set_title(title); ax.set_xlabel(''); ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=40)
    for t in ax.get_xticklabels():
        t.set_ha('right')

fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
genre_heatmap('mood', MOODS, MOOD_RU, 'Жанр × настроение', '', axes[0])
genre_heatmap('ctx', CTXS, CTX_RU, 'Жанр × контекст', '', axes[1])
genre_heatmap('atm', ATMS, ATM_RU, 'Жанр × атмосфера', '', axes[2])
plt.tight_layout(); savefig('05_genre_tag_heatmaps'); plt.show()

fig, ax = plt.subplots(figsize=(11, 5))
inst_cols = [f'inst_{i}' for i in INSTS] + ['vocals_prob']
M = features.groupby('genre')[inst_cols].mean().reindex(genre_order)
M.index = [GENRE_RU[g] for g in M.index]; M.columns = [INST_RU.get(c[5:], 'вокал') for c in inst_cols]
sns.heatmap(M, cmap='Blues', annot=True, fmt='.2f', annot_kws={'size': 7}, linewidths=1, linecolor='#fcfcfb', ax=ax, vmin=0, vmax=1)
ax.set_title('Жанр × инструменты и вокал: P(«есть»)'); ax.set_ylabel('')
plt.tight_layout(); savefig('06_genre_instruments'); plt.show()

### 4.4 Сигнальные признаки по жанрам

In [ ]:
sig_show = [('tempo_bpm', 'Темп, BPM'), ('rms_mean', 'Энергия (RMS)'), ('spectral_centroid_hz', 'Яркость, Гц'),
            ('onset_rate', 'Атак в секунду'), ('pulse_clarity', 'Ритмичность'), ('percussive_ratio', 'Доля ударных')]
fig, axes = plt.subplots(2, 3, figsize=(17, 8))
labels_ru = [GENRE_RU[g] for g in genre_order]
for ax, (col, title) in zip(axes.flat, sig_show):
    data = [features.loc[features.genre == g, col].dropna().values for g in genre_order]
    bp = ax.boxplot(data, vert=False, widths=0.55, patch_artist=True, showfliers=True,
                    medianprops={'color': '#0b0b0b', 'linewidth': 1.5},
                    boxprops={'facecolor': '#cde2fb', 'edgecolor': ACCENT},
                    whiskerprops={'color': '#9b9a95'}, capprops={'color': '#9b9a95'},
                    flierprops={'marker': 'o', 'markersize': 4, 'markerfacecolor': ACCENT, 'markeredgecolor': 'none'})
    ax.set_yticks(range(1, len(genre_order) + 1)); ax.set_yticklabels(labels_ru, fontsize=8)
    ax.invert_yaxis(); ax.set_title(title)
plt.tight_layout(); savefig('07_signal_by_genre'); plt.show()

mode_share = features.groupby('genre')['mode_major'].mean().reindex(genre_order).rename(index=GENRE_RU)
mode_share.round(2).to_frame('доля мажора').T

### 4.5 Корреляции
Беру корреляцию Спирмена: признаки распределены не нормально, и выбросов хватает.

In [ ]:
corr_cols = SIGNAL + [f'mood_{m}' for m in MOODS] + [f'ctx_{c}' for c in CTXS] + ['vocals_prob', 'year', 'log_views']
C = features[corr_cols].corr(method='spearman')
nice = {**{f'mood_{k}': f'настр: {v}' for k, v in MOOD_RU.items()}, **{f'ctx_{k}': f'конт: {v}' for k, v in CTX_RU.items()},
        'tempo_bpm': 'темп', 'rms_mean': 'энергия', 'rms_dynamic_range_db': 'дин. диапазон', 'spectral_centroid_hz': 'яркость',
        'spectral_flatness': 'шумность', 'onset_rate': 'плотн. атак', 'pulse_clarity': 'ритмичность',
        'percussive_ratio': 'доля ударных', 'zcr_mean': 'ZCR', 'mode_major': 'мажор', 'key_strength': 'сила тональности',
        'vocals_prob': 'вокал', 'year': 'год', 'log_views': 'популярность'}
Cn = C.rename(index=nice, columns=nice)
fig, ax = plt.subplots(figsize=(15, 12.5))
mask = np.triu(np.ones_like(Cn, dtype=bool), k=1)
sns.heatmap(Cn, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5, linecolor='#fcfcfb',
            cbar_kws={'shrink': 0.6, 'label': 'Spearman ρ'})
ax.set_title('Корреляции признаков (Spearman)')
plt.tight_layout(); savefig('08_feature_correlations'); plt.show()

# самые сильные связи сигнал - тег
sem_cols = [f'mood_{m}' for m in MOODS] + [f'ctx_{c}' for c in CTXS] + ['vocals_prob']
pairs = C.loc[SIGNAL, sem_cols].stack().rename('rho').reset_index()
pairs.columns = ['сигнал', 'тег', 'rho']
pairs['|rho|'] = pairs['rho'].abs()
top_pairs = pairs.sort_values('|rho|', ascending=False).head(12)
top_pairs['сигнал'] = top_pairs['сигнал'].map(nice); top_pairs['тег'] = top_pairs['тег'].map(nice)
top_pairs[['сигнал', 'тег', 'rho']].round(3)

### 4.6 Жанр от CLAP против настоящего
Если модель нормально угадывает жанр, значит, и остальным её тегам можно хоть как-то доверять.

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(features['genre'], features['clapgenre_top'], labels=genre_order)
fig, ax = plt.subplots(figsize=(8.5, 7))
sns.heatmap(pd.DataFrame(cm, index=labels_ru, columns=labels_ru), annot=True, fmt='d', cmap='Blues', cbar=False,
            linewidths=1, linecolor='#fcfcfb', ax=ax)
ax.set_xlabel('предсказание CLAP (zero-shot)'); ax.set_ylabel('эталонный жанр')
acc = accuracy_score(features['genre'], features['clapgenre_top'])
ax.set_title(f'Матрица ошибок zero-shot жанра, accuracy = {acc:.2f}')
plt.xticks(rotation=40, ha='right'); plt.tight_layout(); savefig('09_zero_shot_genre_confusion'); plt.show()

per_genre_acc = (features.assign(ok=features.genre == features.clapgenre_top)
                 .groupby('genre')['ok'].mean().reindex(genre_order).rename(index=GENRE_RU))
per_genre_acc.round(2).to_frame('accuracy').T

### 4.7 Сводка по жанрам

In [ ]:
summary = []
mood_by_genre = features.groupby('genre')[[f'mood_{m}' for m in MOODS]].mean().reindex(genre_order)
ctx_by_genre = features.groupby('genre')[[f'ctx_{c}' for c in CTXS]].mean().reindex(genre_order)
for g in genre_order:
    m = mood_by_genre.loc[g].idxmax()[5:]; c = ctx_by_genre.loc[g].idxmax()[4:]
    summary.append({'жанр': GENRE_RU[g], 'типичное настроение': MOOD_RU[m], 'типичный контекст': CTX_RU[c],
                    'медианный темп': round(features.loc[features.genre == g, 'tempo_bpm'].median()),
                    'доля с вокалом': round(features.loc[features.genre == g, 'has_vocals'].mean(), 2)})
display(pd.DataFrame(summary))

# энтропия: 1 значит, что все классы встречаются одинаково часто
def norm_entropy(s, k):
    p = s.value_counts(normalize=True).values
    return float(-(p * np.log(p)).sum() / np.log(k))
print('Энтропия top-1 меток:')
for pfx, labs, name in (('mood', MOODS, 'настроение'), ('ctx', CTXS, 'контекст'), ('atm', ATMS, 'атмосфера')):
    print(f'  {name}: {norm_entropy(features[pfx + "_top"], len(labs)):.2f}, средняя уверенность top-1 = {features[pfx + "_top_conf"].mean():.2f}')
print(f'Zero-shot жанр: accuracy {acc:.2f} при случайном уровне {1/len(genre_order):.2f}')
best = top_pairs.iloc[0]
print(f'Самая сильная связь сигнал↔тег: {best["сигнал"]} ↔ {best["тег"]} (ρ = {best["rho"]:.2f})')

## 5. Похожесть треков

Похожесть считаю просто: косинусная близость и k ближайших соседей. Сравниваю несколько вариантов представления трека:
- эмбеддинг CLAP (512);
- семантические теги: настроение, контекст, атмосфера, инструменты, вокал (36 чисел);
- сигнальные признаки librosa (11 чисел);
- гибрид: все три блока, каждый нормирован и взят со своим весом;
- случайные векторы для сравнения.

Жанр от CLAP ни в один вариант не входит.

Данных о прослушиваниях у меня нет, поэтому качество оцениваю косвенно:
- **Genre P@k**: какая доля из k ближайших соседей того же жанра. Случайно получится примерно 0.07.
- **Era P@k**: то же самое для эпохи.
- Слушаю соседей для нескольких треков (ссылки в таблицах).
- Убираю группы признаков по очереди и смотрю, как меняется качество.

In [ ]:
import umap
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline

features = pd.read_csv(DATA_DIR / 'features.csv')
E_all = np.load(DATA_DIR / 'clap_embeddings.npy')
ids = pd.read_csv(DATA_DIR / 'clap_embeddings_ids.csv').iloc[:, 0].tolist()
E = E_all[[ids.index(t) for t in features['track_id']]]
N = len(features)

MOODS = [c for c in features.columns if c.startswith('mood_') and not c.startswith('mood_top')]
CTXS = [c for c in features.columns if c.startswith('ctx_') and not c.startswith('ctx_top')]
ATMS = [c for c in features.columns if c.startswith('atm_') and not c.startswith('atm_top')]
INSTS = [c for c in features.columns if c.startswith('inst_')] + ['vocals_prob']
SEMANTIC = MOODS + CTXS + ATMS + INSTS
SIGNAL = ['tempo_bpm', 'rms_mean', 'rms_dynamic_range_db', 'spectral_centroid_hz', 'spectral_flatness',
          'onset_rate', 'pulse_clarity', 'percussive_ratio', 'zcr_mean', 'mode_major', 'key_strength']

def l2n(X):
    return X / np.clip(np.linalg.norm(X, axis=1, keepdims=True), 1e-9, None)
def zs(X):
    X = np.nan_to_num(np.asarray(X, dtype=float), nan=np.nanmean(X))
    return (X - X.mean(0)) / np.clip(X.std(0), 1e-9, None)

B_clap, B_sem, B_sig = l2n(E), l2n(zs(features[SEMANTIC].values)), l2n(zs(features[SIGNAL].values))
W = {'clap': 1.0, 'sem': 0.7, 'sig': 0.4}

def hybrid(w=W):
    blocks = [B * w[k] for k, B in (('clap', B_clap), ('sem', B_sem), ('sig', B_sig)) if w.get(k, 0) > 0]
    return l2n(np.hstack(blocks))

rng = np.random.default_rng(SEED)
REPS = {
    'CLAP-эмбеддинг': B_clap,
    'Семантические теги': B_sem,
    'Сигнальные признаки': B_sig,
    'Гибрид': hybrid(),
    'Случайный': l2n(rng.normal(size=(N, 32))),
}
genres = features['genre'].to_numpy(dtype=str)
eras = features['era'].to_numpy(dtype=str)

In [ ]:
def neighbors(X, k):
    S = X @ X.T
    np.fill_diagonal(S, -np.inf)
    idx = np.argsort(-S, axis=1)[:, :k]
    return idx, np.take_along_axis(S, idx, axis=1)

def precision_at_k(X, labels, k=5):
    idx, _ = neighbors(X, k)
    return float((labels[idx] == labels[:, None]).mean())

rows = []
for name, X in REPS.items():
    rows.append({'представление': name, 'размерность': X.shape[1],
                 'Genre P@5': precision_at_k(X, genres, 5), 'Genre P@10': precision_at_k(X, genres, 10),
                 'Era P@5': precision_at_k(X, eras, 5)})
results = pd.DataFrame(rows).set_index('представление')
gc = pd.Series(genres).value_counts()
chance_genre = float((gc * (gc - 1)).sum() / (N * (N - 1)))
print(f'случайный уровень: {chance_genre:.3f}')
print(results.round(3))

fig, ax = plt.subplots(figsize=(9, 3.6))
r = results['Genre P@5'].sort_values()
ax.barh(r.index, r.values, color=[ACCENT if n != 'Случайный' else '#c9c8c3' for n in r.index], height=0.6)
ax.axvline(chance_genre, color='#52514e', ls='--', lw=1); ax.text(chance_genre, -0.7, ' случайно', fontsize=8, color='#52514e')
for y, v in enumerate(r.values):
    ax.text(v + 0.01, y, f'{v:.2f}', va='center', fontsize=9)
ax.set_xlim(0, 1); ax.set_xlabel('Genre P@5'); ax.set_title('Genre P@5 по представлениям')
plt.tight_layout(); savefig('10_similarity_precision'); plt.show()

### 5.1 UMAP и t-SNE
Цвет и маркер показывают жанр. Чем компактнее группы, тем лучше представление различает стили.

In [ ]:
proj = {}
for name in ['CLAP-эмбеддинг', 'Семантические теги', 'Гибрид']:
    X = REPS[name]
    proj[(name, 'UMAP')] = umap.UMAP(n_neighbors=10, min_dist=0.3, metric='cosine', random_state=SEED).fit_transform(X)
    proj[(name, 't-SNE')] = TSNE(n_components=2, perplexity=min(15, N // 4), metric='cosine', init='pca',
                                 random_state=SEED).fit_transform(X)

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for j, name in enumerate(['CLAP-эмбеддинг', 'Семантические теги', 'Гибрид']):
    for i, method in enumerate(['UMAP', 't-SNE']):
        scatter_by_genre(axes[i, j], proj[(name, method)], genres, f'{method}: {name}')
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=12, fontsize=9, bbox_to_anchor=(0.5, -0.01))
plt.tight_layout(rect=(0, 0.03, 1, 1)); savefig('11_embedding_space'); plt.show()

In [ ]:
xy = proj[('Гибрид', 'UMAP')]
fig, ax = plt.subplots(figsize=(15, 11))
scatter_by_genre(ax, xy, genres, 'UMAP гибридного представления (подписаны исполнители)')
for (x, y), a in zip(xy, features['artist']):
    ax.annotate(a[:18], (x, y), fontsize=6.5, color='#52514e', xytext=(3, 3), textcoords='offset points')
ax.legend(loc='best', fontsize=8, ncol=2)
plt.tight_layout(); savefig('12_umap_labeled'); plt.show()

# тот же UMAP, но цвет по настроению
fig, ax = plt.subplots(figsize=(9, 7))
moods_top = features['mood_top'].to_numpy(dtype=str)
for i, m in enumerate(sorted(set(moods_top))):
    sel = moods_top == m
    ax.scatter(xy[sel, 0], xy[sel, 1], s=46, c=CAT[i % 8], marker='o' if i < 8 else '^', label=m,
               edgecolors='#fcfcfb', linewidths=1.2)
ax.set_title('UMAP гибрида: цвет — настроение (CLAP)'); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
ax.legend(fontsize=8)
plt.tight_layout(); savefig('13_umap_by_mood'); plt.show()

### 5.2 Ближайшие треки
5 соседей в гибридном пространстве для нескольких треков. По ссылкам можно послушать и проверить.

In [ ]:
def show_neighbors(track_id, rep='Гибрид', k=5):
    X = REPS[rep]
    i = features.index[features.track_id == track_id][0]
    idx, sims = neighbors(X, k)
    q = features.loc[i]
    print(f"▶ {q.artist} — {q.title}  [{GENRE_RU[q.genre]}, {q.year}, настроение: {q.mood_top}, контекст: {q.ctx_top}]")
    out = features.loc[idx[i], ['artist', 'title', 'genre', 'year', 'mood_top', 'ctx_top', 'clip_url']].copy()
    out.insert(0, 'сходство', sims[i].round(3))
    out['genre'] = out['genre'].map(GENRE_RU)
    display(out.reset_index(drop=True))

query_ids = ['rock_03', 'electronic_01', 'jazz_02', 'classical_05', 'hip_hop_02', 'ambient_05', 'country_02', 'metal_01']
for t in query_ids:
    if t in set(features.track_id):
        show_neighbors(t)

In [ ]:
# один и тот же трек в разных представлениях
t = 'jazz_02' if 'jazz_02' in set(features.track_id) else features.track_id.iloc[0]
for rep in ['CLAP-эмбеддинг', 'Семантические теги', 'Сигнальные признаки']:
    print(f'\n=== {rep} ===')
    show_neighbors(t, rep=rep, k=5)

### 5.3 Похожие артисты
У каждого исполнителя один трек, так что похожие артисты здесь просто авторы трёх ближайших треков.

In [ ]:
idx, sims = neighbors(REPS['Гибрид'], 3)
features['similar_artists_emb'] = ['|'.join(features.loc[row, 'artist']) for row in idx]
sim_art = features[['artist', 'genre', 'similar_artists_emb']].copy()
if 'llm_similar_artists' in features.columns:
    sim_art['llm_similar_artists'] = features['llm_similar_artists']
sim_art.to_csv(DATA_DIR / 'similar_artists.csv', index=False)
sim_art['genre'] = sim_art['genre'].map(GENRE_RU)
sim_art.sample(12, random_state=SEED)

### 5.4 Какие признаки полезнее
1. Убираю из гибрида по одному блоку и смотрю, насколько падает Genre P@5.
2. Считаю kNN отдельно по каждой группе признаков.
3. Обучаю Random Forest предсказывать жанр (кросс-валидация на 4 фолда) и смотрю важность признаков. Для сравнения логистическая регрессия на эмбеддинге.

In [ ]:
abl = [{'конфигурация': 'полный гибрид', 'Genre P@5': precision_at_k(hybrid(), genres), 'Era P@5': precision_at_k(hybrid(), eras)}]
for k, name in (('clap', 'без CLAP-эмбеддинга'), ('sem', 'без семантических тегов'), ('sig', 'без сигнальных признаков')):
    w = {**W, k: 0}
    abl.append({'конфигурация': name, 'Genre P@5': precision_at_k(hybrid(w), genres), 'Era P@5': precision_at_k(hybrid(w), eras)})
abl = pd.DataFrame(abl).set_index('конфигурация')
abl['Δ Genre P@5'] = abl['Genre P@5'] - abl.loc['полный гибрид', 'Genre P@5']
print(abl.round(3))

groups = {'настроение': MOODS, 'контекст': CTXS, 'атмосфера': ATMS, 'инструменты + вокал': INSTS,
          'темп + энергия': ['tempo_bpm', 'rms_mean', 'onset_rate', 'pulse_clarity'],
          'тембр (яркость, шумность, ZCR)': ['spectral_centroid_hz', 'spectral_flatness', 'zcr_mean'],
          'популярность': ['log_views']}
grp = pd.DataFrame([{'группа признаков': g, 'число признаков': len(cols),
                     'Genre P@5': precision_at_k(l2n(zs(features[cols].values)), genres),
                     'Era P@5': precision_at_k(l2n(zs(features[cols].values)), eras)} for g, cols in groups.items()])
grp = grp.sort_values('Genre P@5', ascending=False).set_index('группа признаков')
grp.round(3)

In [ ]:
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)
X_int = np.nan_to_num(features[SEMANTIC + SIGNAL].values.astype(float))
rf = RandomForestClassifier(n_estimators=500, random_state=SEED, min_samples_leaf=2)
acc_sets = {
    'RF: семантика + сигнал': cross_val_score(rf, X_int, genres, cv=cv).mean(),
    'RF: только семантика': cross_val_score(rf, np.nan_to_num(features[SEMANTIC].values), genres, cv=cv).mean(),
    'RF: только сигнал': cross_val_score(rf, np.nan_to_num(features[SIGNAL].values), genres, cv=cv).mean(),
    'LogReg: CLAP-эмбеддинг': cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.5)),
                                              E, genres, cv=cv).mean(),
}
print(pd.Series(acc_sets, name='accuracy').round(3))

rf.fit(X_int, genres)
imp = pd.Series(rf.feature_importances_, index=SEMANTIC + SIGNAL).sort_values(ascending=True).tail(15)
fig, ax = plt.subplots(figsize=(9, 5.5))
colors = [CAT[1] if c in SIGNAL else ACCENT for c in imp.index]
ax.barh(imp.index, imp.values, color=colors, height=0.6)
ax.set_title('Важность признаков (Random Forest)'); ax.set_xlabel('importance')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=ACCENT, label='CLAP'), Patch(color=CAT[1], label='librosa')],
          loc='lower right', fontsize=8)
plt.tight_layout(); savefig('14_feature_importance'); plt.show()

### 5.5 Итог по экспериментам

In [ ]:
best_rep = results.drop('Случайный')['Genre P@5'].idxmax()
most_important_block = abl['Δ Genre P@5'].drop('полный гибрид').idxmin()
print(f"лучшее представление: {best_rep} ({results.loc[best_rep, 'Genre P@5']:.2f})")
print(f"сильнее всего падает: {most_important_block} ({abl.loc[most_important_block, 'Δ Genre P@5']:+.2f})")
print(f"лучшая группа по жанру: {grp.index[0]} ({grp.iloc[0]['Genre P@5']:.2f})")
print(f"лучшая группа по эпохе: {grp['Era P@5'].idxmax()} ({grp['Era P@5'].max():.2f})")
print('топ-5 признаков:', ', '.join(imp.sort_values(ascending=False).index[:5]))

exp = {'representations': results.round(4).to_dict(), 'ablation': abl.round(4).to_dict(),
       'feature_groups': grp.round(4).to_dict(), 'genre_cv_accuracy': {k: round(v, 4) for k, v in acc_sets.items()},
       'top_features': imp.sort_values(ascending=False).round(4).to_dict(), 'hybrid_weights': W}
(DATA_DIR / 'experiment_results.json').write_text(json.dumps(exp, ensure_ascii=False, indent=2), encoding='utf-8')
features.to_csv(DATA_DIR / 'features.csv', index=False)


## 6. Выводы

**Про прогон.** Полностью запустить ноутбук на всех 96 треках я не успела: не хватило времени и нет нормального оборудования (своей видеокарты нет, а на CPU извлечение эмбеддингов CLAP и загрузка всех треков идут очень долго). Поэтому ниже не итоговые цифры, а то, что я ожидаю получить, исходя из того, как устроены модели и признаки. Код написан полностью, после запуска все графики и метрики посчитаются сами.

### Что ожидаю увидеть

**EDA:**
- Теги должны совпадать с очевидными вещами: у метала и хип-хопа больше «aggressive» и «energetic», у эмбиента и классики «calm», «dreamy» и контексты «сон» и «работа/учёба», у электроники и латино «вечеринка».
- Темп и громкость должны коррелировать с «energetic» и «тренировкой» положительно, а со «спокойным» и «сном» отрицательно. Если так и выйдет, значит теги CLAP действительно опираются на звук.
- Жанр от CLAP, думаю, будет угадываться заметно лучше случайного (1/12). Путаница, скорее всего, между близкими жанрами: рок и метал, R&B и хип-хоп, эмбиент и классика.
- Лад (мажор/минор) с «happy»/«sad», скорее всего, будет связан слабо: такая оценка лада по хромаграмме довольно грубая.

**Похожесть треков:**
- Лучше всего по жанру должны работать эмбеддинги CLAP, потому что они кодируют тембр, инструменты и продакшн. Сигнальные признаки по отдельности будут слабее всего: темп 120 BPM бывает и в хаусе, и в кантри.
- Семантические теги, скорее всего, будут хуже эмбеддингов по жанровой точности, но зато будут находить похожие треки по настроению и ситуации из разных жанров.
- Гибрид, по идее, должен быть не хуже чистого CLAP, а по эпохе может быть немного лучше.
- Среди отдельных признаков жду, что самыми полезными окажутся признаки, связанные с энергией: energetic/calm/aggressive, контексты «тренировка» и «сон», доля ударных, яркость звука. Тонкие теги вроде «ностальгический» или «романтический» будут шумнее.

### Что это значит для рекомендаций
- Аудио-эмбеддинги стоит брать как основу контентной части. Особенно они полезны для новых треков без прослушиваний.
- Теги настроения и контекста удобны для плейлистов под ситуацию и для объяснения рекомендаций, но сами по себе для похожести хуже эмбеддингов.
- Темп и энергию можно использовать как фильтры (например, для тренировки).
- Год и популярность почти ничего не говорят о звучании, но нужны при ранжировании, чтобы смешивать хиты и менее известные треки.
- В реальной системе всё это нужно объединять с данными о прослушиваниях.

### Ограничения и проблемы
- Датасет маленький, по 8 треков на жанр, так что метрики будут шумными.
- 30 секунд не описывают трек целиком, особенно длинные треки, которые сильно меняются по ходу (Bohemian Rhapsody, Stairway to Heaven).
- Zero-shot CLAP зависит от формулировок промптов и любит некоторые классы больше других. Поправка на среднее частично это исправляет, но одновременно выравнивает распределение тегов.
- Для контекстов нет никакой разметки, проверить их можно только косвенно.
- Жанр как мера похожести слишком строгий: треки разных жанров могут звучать похоже.
- На YouTube можно попасть на ремастер или клип со вставками, поэтому автоподбор видео приходится проверять руками. Просмотры зависят от возраста видео.
- librosa часто ошибается с темпом в 2 раза (60 вместо 120).
- У классики год означает год написания, а не записи.

### Что можно доделать
Довести полный прогон и сравнить реальные цифры с ожиданиями; увеличить датасет; вручную разметить часть контекстов; попробовать MERT и модели Essentia вместо CLAP; добавить данные о прослушиваниях.